# 03 — Features & Modelling

Builds features (`src/features.py`), tunes 4 model families + 2 baselines with time-ordered CV, selects on validation, evaluates **once** on the test years, runs the storm-weighting ablation, and saves the production model.

| Split | Years | Role |
|---|---|---|
| train | 1998–2016 | fitting + `TimeSeriesSplit(5)` CV tuning |
| validation | 2017–2019 | model selection (note: solar minimum) |
| test | 2020–2026-01 | single final evaluation (includes May 2024 storm) |

**Run with kernel "Python 3.12 (tormenta)".** Full run ≈ 30–60 min on this laptop; set `FAST = True` for a ~2-minute smoke test (results not meaningful). Close other heavy apps first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, train, evaluate as ev
from src.features import derived
from src.predict import predict_with_interval
from src.preprocessing import load_processed

FAST = False          # True = quick smoke test only
config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)

## 1. Features
One call to the shared `build_features` — the same function the live apps use.

In [ ]:
X, y, persistence = train.load_xy()
masks = train.split_masks(X.index)

print(f"{X.shape[1]} features")
print({k: f"{v.sum():,} hours" for k, v in masks.items()})
groups = {
    "raw": config.SW_FEATURES,
    "derived": ["pressure", "efield", "vbs", "beta", "mach_a", "bt", "sin_half_clock", "newell"],
    "lags": [c for c in X if "_lag" in c],
    "rolling": [c for c in X if any(s in c for s in ("_mean", "_std", "_min", "_max")) and not c.startswith("f107")],
    "F10.7": [c for c in X if c.startswith("f107")],
    "time": [c for c in X if c.startswith(("hour_", "doy_"))],
}
pd.DataFrame({"count": {g: len(c) for g, c in groups.items()}, "columns": {g: ", ".join(c) for g, c in groups.items()}})

### 1b. Check: our recomputed physics vs OMNI's own columns
β and Alfvén Mach use OMNI's formulas, so they should match closely. Our pressure omits the alpha-particle term (SWPC has no alpha data), so expect a near-constant ratio. After this check OMNI's columns are never used again.

In [ ]:
raw = load_processed()
sample = raw.loc["2005":"2006"].dropna()
d = derived(sample[config.SW_FEATURES])
check = pd.DataFrame({
    "ours / OMNI (median)": [(d[a] / sample[b]).median() for a, b in
                             [("pressure", "omni_pressure"), ("efield", "omni_efield"),
                              ("beta", "omni_beta"), ("mach_a", "omni_mach_a")]],
    "correlation": [d[a].corr(sample[b]) for a, b in
                    [("pressure", "omni_pressure"), ("efield", "omni_efield"),
                     ("beta", "omni_beta"), ("mach_a", "omni_mach_a")]],
}, index=["pressure", "efield", "beta", "mach_a"])
check

## 2. Tuning (CV within the training years)
`RandomizedSearchCV` + `TimeSeriesSplit(5)` → each fold trains on the past and validates on the future. KNN/SVR use every 9th hour (kernel SVR is O(n²–n³)); RF is tuned on every 3rd hour and refit on all.

In [ ]:
tuned = train.tune_models(X, y, masks, fast=FAST)

## 3. Validation: model selection + over/under-fitting
`train_rmse` ≪ `cv_rmse` signals overfitting; all three high and similar signals underfitting. Validation (2017–2019) is solar minimum, so we read **CV RMSE, validation RMSE and storm RMSE together**.

In [ ]:
val_table = train.validation_table(tuned, X, y, masks, persistence)
val_table.to_csv(config.METRICS_DIR / "validation.csv")
display(val_table.drop(columns="n"))
m = val_table.drop(index="Persistence (baseline)")
ev.plot_overfitting(m["train_rmse"], m["cv_rmse"], m["rmse"], "09_overfitting")
plt.show()

## 4. Test set — evaluated once
Each family is refit on train+val with its tuned settings.

In [ ]:
test_table, test_preds, test_models = train.test_evaluation(tuned, X, y, masks, persistence, fast=FAST)
test_table.to_csv(config.METRICS_DIR / "test.csv")
display(test_table.drop(columns="n"))
ev.plot_model_comparison(test_table, "Test 2020–2026", "10_model_comparison")
plt.show()

In [ ]:
y_test = y[masks["test"]]
levels = ev.error_by_level(y_test, test_preds)
levels.to_csv(config.METRICS_DIR / "test_error_by_level.csv")
display(levels)
ev.plot_error_by_level(levels, "11_error_by_level")
plt.show()
ev.plot_pred_vs_actual(y_test, test_preds["HistGB"], "HistGB (test)", "12_pred_vs_actual")
plt.show()

## 5. Ablation: storm weighting (w = 1 + Kp/3)
Does up-weighting storm hours reduce storm under-prediction, and at what cost overall?

In [ ]:
abl_table, abl_preds = train.weighting_ablation(tuned["HistGB"]["best_params"], X, y, masks, fast=FAST)
abl_table.to_csv(config.METRICS_DIR / "ablation_weighting.csv")
display(abl_table.drop(columns="n"))
ev.error_by_level(y_test, abl_preds)

## 6. What does the model rely on?
Permutation importance of the train-only HistGB on the validation years (how much RMSE worsens when one feature is shuffled).

In [ ]:
val = masks["val"]
_, importance = ev.plot_permutation_importance(tuned["HistGB"]["estimator"], X[val], y[val], "13_feature_importance")
plt.show()
importance.head(15).round(4)

## 7. Case study: May 2024 storm (model trained only on 1998–2019)

In [ ]:
case = train.fit_bundle(tuned["HistGB"]["best_params"], X, y, masks["train"] | masks["val"], fast=FAST)
band = predict_with_interval(case, X[masks["test"]])
ev.plot_storm_case(y_test, band["kp_pred"], band["kp_lo"], band["kp_hi"], "2024-05-09", "2024-05-14",
                   "May 2024 storm: HistGB nowcast vs observed Kp (never seen in training)", "14_may2024_nowcast")
plt.show()
cov = ((y_test >= band["kp_lo"]) & (y_test <= band["kp_hi"])).mean()
print(f"80% interval coverage on test: {cov:.1%}")

## 8. Production model
HistGB (chosen for accuracy, small size, native NaN handling, quantile intervals) trained on all definitive data **up to the start of the last 6 months**; those 6 months are the validation-gate holdout that the weekly retrain will use. Saved as `models/tormenta_vYYYYMMDD.joblib` + `models/current.joblib`.

In [ ]:
bundle, gate = train.fit_production(tuned["HistGB"]["best_params"], X, y, fast=FAST)
if FAST:
    print("FAST mode: not saving the model.")
else:
    version = train.save_bundle(bundle, test_table.loc["HistGB"].to_dict(), gate)
    print("Saved", version)
pd.Series(gate)